In [ ]:
# Cell 1: Install arc-agi from competition wheels (offline, works in Phase A and B)
import subprocess, sys
wheels = '/kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels'
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    '--no-index', f'--find-links={wheels}',
    'arc-agi', 'python-dotenv'
])
print('[Cell1] arc-agi installed from competition wheels')

In [ ]:
# Cell 2: Write VericodingAgent to /tmp/my_agent.py
agent_code = '''import random, itertools, collections
import numpy as np
from agents.agent import Agent
from arcengine import GameAction


class VericodingAgent(Agent):
    """Graph exploration agent with D4 Zobrist hash."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._step = 0
        # State graph: state_hash -> {action_key -> next_state_hash or None}
        self._graph = collections.defaultdict(dict)
        self._visited = set()
        self._last_hash = None
        self._last_key = None
        self._stall = 0
        # D4 Zobrist table (64x64x16 uint64)
        rng = np.random.RandomState(42)
        self._zob = rng.randint(0, 2**63-1, size=(64, 64, 16), dtype=np.uint64)
        print(f'[V25] agent init | D4 Zobrist {self._zob.shape}')

    def _hash(self, grid):
        """D4 Zobrist: min XOR over 8 dihedral transforms."""
        h, w = grid.shape[:2]
        best = np.uint64(2**64-1)
        for rot in range(4):
            for flip in (False, True):
                g = np.rot90(grid, rot)
                if flip: g = np.fliplr(g)
                gh, gw = g.shape[:2]
                mask = self._zob[:gh, :gw, :] * (g[:, :, None] > 0)
                hv = np.bitwise_xor.reduce(mask, axis=(1, 2))
                hv_int = int(np.bitwise_xor.reduce(hv))
                if hv_int < int(best):
                    best = np.uint64(hv_int)
        return int(best)

    def _action_key(self, action_idx, frame):
        """Distinguish click actions by coordinates."""
        if hasattr(frame, 'available_actions') and frame.available_actions:
            if action_idx < len(frame.available_actions):
                a = frame.available_actions[action_idx]
                if hasattr(a, 'is_complex') and a.is_complex():
                    # Use (action_idx, x, y) from last click data if available
                    return (action_idx,)
        return action_idx

    def choose_action(self, frames, latest_frame) -> GameAction:
        try:
            return self._fsm(frames, latest_frame)
        except Exception as e:
            print(f'V25 err: {e}')
            return self._fallback(latest_frame)

    def _fsm(self, frames, latest_frame) -> GameAction:
        self._step += 1
        grid = np.array(latest_frame.frame[0], dtype=np.int32)
        avail = list(latest_frame.available_actions) if latest_frame.available_actions else []
        if not avail:
            return GameAction(1)
        h, w = grid.shape[:2]

        # Hash current state
        cur_hash = self._hash(grid)
        is_new = cur_hash not in self._visited
        self._visited.add(cur_hash)

        # Record transition from previous state
        if self._last_hash is not None and self._last_key is not None:
            if cur_hash not in self._graph[self._last_hash].get(self._last_key, set()):
                self._graph[self._last_hash][self._last_key] = cur_hash
            # If state changed, action was productive = reset stall
            if cur_hash != self._last_hash:
                self._stall = 0

        self._last_hash = cur_hash

        # --- Frontier exploration ---
        # From current state, find an action that leads to UNVISITED state
        if cur_hash in self._graph:
            for act_key, next_h in self._graph[cur_hash].items():
                if isinstance(act_key, tuple):
                    act_idx = act_key[0]
                else:
                    act_idx = act_key
                if next_h not in self._visited:
                    # Found frontier action
                    self._last_key = act_key
                    return self._make_action(act_idx, grid, latest_frame)

            # All explored from here: try untried actions
            tried_keys = set(self._graph[cur_hash].keys())
            untried = [a for a in avail if self._action_key(a, latest_frame) not in tried_keys]
            if untried:
                ch = untried[0]
                self._last_key = self._action_key(ch, latest_frame)
                return self._make_action(ch, grid, latest_frame)

            # Fully explored: random action
            ch = random.choice(avail)
            self._last_key = self._action_key(ch, latest_frame)
            return self._make_action(ch, grid, latest_frame)

        # --- First visit to this state: systematic exploration ---
        # Separate complex (click) and simple actions
        complex_a = []
        simple_a = []
        for a in avail:
            if isinstance(a, int):
                simple_a.append(a)
            elif hasattr(a, 'is_complex') and a.is_complex():
                complex_a.append(a)
            else:
                simple_a.append(a)

        nz = np.count_nonzero(grid)

        if complex_a and nz > 0:
            # Click on non-zero cell
            flat = grid.flatten()
            nz_idx = np.where(flat > 0)[0]
            idx = nz_idx[self._step % len(nz_idx)]
            y, x = divmod(int(idx), w)
            a = GameAction(6)
            a.set_data({'x': x, 'y': y})
            self._last_key = (6, x, y)
            return a

        if simple_a:
            ch = simple_a[self._step % len(simple_a)]
            self._last_key = ch
            return self._make_action(ch, grid, latest_frame)

        return self._fallback(latest_frame)

    def _make_action(self, idx, grid, frame):
        a = GameAction(idx)
        if hasattr(a, 'is_complex') and a.is_complex():
            y = min(int(grid.shape[0]) - 1, max(0, random.randint(0, grid.shape[0] - 1))) if grid.shape[0] > 0 else 0
            x = min(int(grid.shape[1]) - 1, max(0, random.randint(0, grid.shape[1] - 1))) if grid.shape[1] > 0 else 0
            a.set_data({'x': x, 'y': y})
            self._last_key = (idx, x, y)
        else:
            self._last_key = idx
        return a

    def _fallback(self, frame) -> GameAction:
        avail = list(getattr(frame, 'available_actions', None) or [])
        if not avail:
            return GameAction(1)
        idx = avail[self._step % len(avail)]
        return self._make_action(idx, getattr(frame, 'frame', [[0]]), frame)
'''

with open("/tmp/my_agent.py", "w") as f:
    f.write(agent_code)
print('[Cell2] V25 graph agent written')


In [ ]:
# Cell 3: Phase B — Competition Rerun (gateway + framework)
import os, subprocess, shutil, sys

if os.environ.get("KAGGLE_IS_COMPETITION_RERUN"):
    print("[Cell3] Phase B: competition rerun detected")

    # Wait for gateway sidecar
    subprocess.run([
        "curl", "--fail", "--retry", "999",
        "--retry-delay", "1", "--max-time", "3",
        "http://gateway:8001/api/games"
    ])

    # Copy framework boilerplate to working dir
    src = "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/agents"
    if os.path.isdir(src):
        shutil.copytree(src, "/kaggle/working/agents", dirs_exist_ok=True)
        print("[Cell3] framework copied")
    else:
        print(f"[Cell3] WARNING: {src} not found")

    # Read our agent code written by Cell 2, write into framework location
    with open("/tmp/my_agent.py") as f:
        agent_code = f.read()
    with open("/kaggle/working/agents/my_agent.py", "w") as f:
        f.write(agent_code)

    # Write __init__.py with proper agent registration
    init = """from agents.my_agent import VericodingAgent

AVAILABLE_AGENTS = {
    "vericoding": VericodingAgent,
}
"""
    with open("/kaggle/working/agents/__init__.py", "w") as f:
        f.write(init)
    print("[Cell3] agent registered")

    # Run framework main.py
    os.chdir("/kaggle/working")
    subprocess.run([
        sys.executable, "main.py",
        "--agent", "vericoding",
        "--output", "/kaggle/working/submission.parquet",
    ])
    print("[Cell3] Phase B complete")
else:
    print("[Cell3] Phase A: skipping (no competition rerun)")


In [ ]:
# Cell 4: Phase A — Write dummy submission.parquet (required to enable Submit button)
import os
import pandas as pd

if not os.environ.get('KAGGLE_IS_COMPETITION_RERUN'):
    print('[Cell4] Phase A: writing dummy submission.parquet')
    pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score']
    ).to_parquet('/kaggle/working/submission.parquet', index=False)
    print('[Cell4] submission.parquet written — click Submit to Competition on Kaggle')
else:
    print('[Cell4] Phase B: submission.parquet handled by framework gateway')